# 原 Stage1 0bp 完整账户：逐日闭合的 SOXX 相对贡献

复用原生Qlib账本、原价和公司行动；方法、来源、预算与非因果边界见 plan.json。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.analysis.etf_transfer_attribution import holdings_attribution
from src.analysis.integrated_allocation import paired_metrics
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/stage1_original_0bp_account_contributions_20261023'
plan=json.loads((card/'plan.json').read_text());src=plan['sources']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
assert plan['budget']=={'new_market_requests':0,'helper_calls_max':1,'source_Recorder_object_loads_max':2,'holdings_attribution_calls_max':1,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Qlib_recorders':1}
assert sha(root/src['stage1_card']/'summary.json')==src['stage1_summary_sha256']
assert sha(root/src['stage1_card']/'verification_manifest.json')==src['stage1_manifest_sha256']
assert sha(root/src['account_helper_card']/'account_source_preflight.json')==src['account_helper_receipt_sha256']
assert sha(root/src['provider_uri']/'qlib_store_manifest.json')==src['provider_manifest_sha256']
assert sha(root/'src/analysis/etf_transfer_attribution.py')==src['corrected_attribution_module_sha256']
assert sha(root/'research/stage1_beta_composition_account_contributions_20261023/component_result.json')==src['separate_composition_component_result_sha256']
source_summary=json.loads((root/src['stage1_card']/'summary.json').read_text())
source_manifest=json.loads((root/src['stage1_card']/'verification_manifest.json').read_text())
source_binding=json.loads((root/src['stage1_card']/'nas_binding.json').read_text())
assert source_summary['recorder_id']==src['stage1_recorder_id'] and source_binding['evidence']['sha256']==source_summary['nas_evidence']['sha256']==src['stage1_NAS_archive_sha256']
for artifact,key in [('native/A_active_research_replay_0','stage1_native_artifact_sha256'),('paths/A_active_research_replay_0','stage1_path_artifact_sha256')]:
    matches=[root/src['stage1_card']/e['path'] for e in source_manifest['entries'] if e['path'].endswith('/'+artifact)]
    assert len(matches)==1 and sha(matches[0])==src[key],artifact
account=prepare_account_sources(root=root,card=root/src['account_helper_card'])
assert sha(root/src['account_helper_card']/'account_source_preflight.json')==src['account_helper_receipt_sha256']
qlib.init(provider_uri=str(root/src['provider_uri']),region='us')
assert len(D.calendar(start_time='2023-01-03',end_time='2026-10-09',freq='day'))>=944
with R.uri_context(str(root/src['stage1_card']/'qlib_recorder')):
    recorder=R.get_recorder(experiment_name=Path(src['stage1_card']).name,recorder_id=src['stage1_recorder_id'])
    book=recorder.load_object('native/A_active_research_replay_0')
    path=recorder.load_object('paths/A_active_research_replay_0')
assert book['audit']['status']=='passed' and len(book['authority'])==len(path['replay'])==944
assert len(book['points'])==len(account['points'])==3778
for role,other in [('Stage1_A',book),('SOXX',account['references']['SOXX'])]:
    pd.testing.assert_frame_equal(other['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True),account['references'][role]['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True))
assert abs(book['authority'].native_expected_nav.iloc[-1]-account['references']['Stage1_A']['authority'].native_expected_nav.iloc[-1])<1e-9
state={'status':'holdings_attribution_attempted','plan_sha256':sha(card/'plan.json'),'helper_calls':1,'source_recorder_object_loads':2,'holdings_attribution_calls':1,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0}
atomic_json(state,card/'run_state.json')
raw_targets=account['frame'].set_index('signal_date')
daily,segments=holdings_attribution(book,account['points'],account['schedule'],raw_targets,initial_cash=100000)
assert len(daily)==944 and len(segments)==944*2*4 and daily.closure_error.abs().max()<1e-9
ref=account['references']['SOXX']['authority'].reset_index(drop=True)
assert pd.DatetimeIndex(ref.trade_date).equals(pd.DatetimeIndex(daily.trade_date))
previous=np.r_[100000.,book['authority'].native_expected_nav.to_numpy(float)[:-1]]
assert np.isfinite(previous).all() and (previous>0).all()
post=pd.DatetimeIndex(daily.trade_date)+pd.Timedelta(hours=9,minutes=47)
points=book['points'].set_index('stamp');prices=account['points']
assert post.isin(points.index).all() and post.isin(prices.index).all()
marks=points.loc[post]
w_soxl=marks.q_soxl.to_numpy(float)*prices.loc[post,'SOXL'].to_numpy(float)/previous
w_cash=(marks.cash.to_numpy(float)+marks.receivable.to_numpy(float))/previous
assert np.isfinite(w_soxl).all() and np.isfinite(w_cash).all() and (w_cash>=-1e-9).all()
candidate=daily['return'].to_numpy(float);benchmark=ref.strategy_return.to_numpy(float)
assert (candidate>-1).all() and (benchmark>-1).all()
cash=-w_cash*benchmark
selection=daily.SOXL_contribution.to_numpy(float)-w_soxl*benchmark
fees=daily.fee_contribution.to_numpy(float)
relative=candidate-benchmark
residual=relative-cash-selection-fees
scale=np.divide(np.log1p(candidate)-np.log1p(benchmark),relative,out=1/(1+benchmark),where=np.abs(relative)>1e-12)
paired=np.log1p(candidate)-np.log1p(benchmark)
components=pd.DataFrame({'signal_date':book['authority'].signal_date,'trade_date':daily.trade_date,'exit_date':daily.exit_date,'Stage1_return':candidate,'SOXX_return':benchmark,'paired_log_excess':paired,'post_open_cash_weight':w_cash,'post_open_soxl_weight':w_soxl,'cash_opportunity_log':cash*scale,'SOXL_selection_log':selection*scale,'Stage1_direct_fees_log':fees*scale,'timing_and_benchmark_residual_log':residual*scale,'risk_state':daily.risk_state})
cols=['cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']
assert np.max(np.abs(components[cols].sum(axis=1)-components.paired_log_excess))<1e-12
relative_metrics=paired_metrics(book['authority'],ref)
assert abs(components.paired_log_excess.sum()-relative_metrics['log_excess_sum'])<1e-9
assert abs(relative_metrics['relative_wealth']-0.6860064369263443)<1e-9

def describe(g,kind,period,partial=False):
    return {'kind':kind,'period':str(period),'partial_boundary_period':bool(partial),'sessions':len(g),'first_trade':str(pd.Timestamp(g.trade_date.iloc[0]).date()),'last_exit':str(pd.Timestamp(g.exit_date.iloc[-1]).date()),'relative_log_excess':float(g.paired_log_excess.sum()),'relative_wealth':float(np.exp(g.paired_log_excess.sum())),**{k:float(g[k].sum()) for k in cols},'mean_post_open_cash_weight':float(g.post_open_cash_weight.mean()),'mean_post_open_soxl_weight':float(g.post_open_soxl_weight.mean()),'risk_state_ge2_fraction':float(g.risk_state.ge(2).mean())}
period_rows=[describe(components,'all','all')]
for freq,kind in [('Y','year'),('Q','quarter')]:
    for period,g in components.groupby(components.trade_date.dt.to_period(freq),sort=True):
        partial=(period==components.trade_date.iloc[0].to_period(freq) or period==components.trade_date.iloc[-1].to_period(freq))
        period_rows.append(describe(g,kind,period,partial))
windows=[describe(components.iloc[i:i+126],'rolling126',components.trade_date.iloc[i]) for i in range(len(components)-125)]
periods=pd.DataFrame(period_rows);rolling=pd.DataFrame(windows)
assert len(rolling)==819 and sum(periods.kind.eq('quarter'))==16 and sum(periods.kind.eq('year'))==4
assert np.max(np.abs(periods[cols].sum(axis=1)-periods.relative_log_excess))<1e-10
assert np.max(np.abs(rolling[cols].sum(axis=1)-rolling.relative_log_excess))<1e-10
full=periods.iloc[0]
separate=json.loads((root/'research/stage1_beta_composition_account_contributions_20261023/component_result.json').read_text())
assert separate['daily_rows']==944 and separate['full_relative_wealth']>1
report={'status':'original_Stage1_native_account_components_closed_descriptive_only','source_recorder_id':src['stage1_recorder_id'],'plan_sha256':sha(card/'plan.json'),'daily_rows':944,'quarters':16,'rolling126_windows':819,'full_relative_log_excess':float(full.relative_log_excess),'full_relative_wealth':float(full.relative_wealth),'full_log_components':{k:float(full[k]) for k in cols},'max_native_daily_return_closure_error':float(daily.closure_error.abs().max()),'max_daily_log_component_closure_error':float(np.max(np.abs(components[cols].sum(axis=1)-components.paired_log_excess))),'residual_absolute_magnitude_relative_to_paired_log':float(components.timing_and_benchmark_residual_log.abs().sum()/components.paired_log_excess.abs().sum()),'negative_quarters':periods.loc[(periods.kind=='quarter')&(periods.relative_log_excess<0),'period'].tolist(),'worst_126':rolling.loc[rolling.relative_log_excess.idxmin()].to_dict(),'separate_composition_context':{'source_component_result_sha256':src['separate_composition_component_result_sha256'],'relative_wealth':separate['full_relative_wealth'],'full_log_components':separate['full_log_components'],'comparison_is_not_causal_policy_effect':True},'no_causal_switch_identified':True,'new_fits':0,'new_accounts':0,'new_bootstraps':0}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path='inputs',plan=plan,source_receipt=account['receipt'],source_identities=src)
    rec.save_objects(artifact_path='analysis',daily=components,segments=segments,periods=periods,rolling126=rolling,component_result=report)
    rec.log_metrics(native_sessions=944,rolling_windows=819,new_model_fits=0,new_native_accounts=0)
    recorder_id,experiment_id=rec.id,str(rec.experiment_id)
summary={'experiment':plan['experiment'],'decision':'descriptive_original_Stage1_component_attribution_no_promotion','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'component_result.json','new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/segments','analysis/rolling126']},'limitations':['Cash/selection/fee terms are descriptive account contributions with explicit residual, not actionable or causal factors.','Original Stage1 and composition candidate have different own states and their component totals cannot be subtracted as an isolated policy effect.','Historical accounts and available raw marks do not qualify independent forward or full-minute risk.']}
atomic_json(report,card/'component_result.json');atomic_json(summary,card/'summary.json')
binding=json.loads((root/src['stage1_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-09')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json');state.update(status='completed',recorder_id=recorder_id,experiment_id=experiment_id)
atomic_json(state,card/'run_state.json')
q={row.period:row for row in periods.loc[periods.kind.eq('quarter')].itertuples()}
worst=report['worst_126']
(card/'conclusion.md').write_text(
    '# 原 Stage1 0bp 账户：现金、SOXL选择与费用贡献\n\n'
    f'只读原生Qlib A_0账户及SOXX同日账户，原始股数、四段原价、应收分红和Tiered费用逐日闭合，944日账户收益最大闭合误差{report["max_native_daily_return_closure_error"]:.2e}，对SOXX对数分量最大闭合误差{report["max_daily_log_component_closure_error"]:.2e}。0拟合、0新账户、0重放。\n\n'
    f'全程原Stage1/SOXX财富比{report["full_relative_wealth"]:.6f}，对数超额{report["full_relative_log_excess"]:+.6f}。分量：现金机会{full.cash_opportunity_log:+.6f}、SOXL相对选择{full.SOXL_selection_log:+.6f}、直接费用{full.Stage1_direct_fees_log:+.6f}、时点/基准残差{full.timing_and_benchmark_residual_log:+.6f}。这些是物理持仓的描述性分量，不是事前选时技能或单独政策反事实。\n\n'
    f'原路径最差126日为{worst["first_trade"]}至{worst["last_exit"]}，相对SOXX对数超额{worst["relative_log_excess"]:+.6f}；现金项{worst["cash_opportunity_log"]:+.6f}、SOXL项{worst["SOXL_selection_log"]:+.6f}、费用{worst["Stage1_direct_fees_log"]:+.6f}、残差{worst["timing_and_benchmark_residual_log"]:+.6f}。对照2024Q3现金{q["2024Q3"].cash_opportunity_log:+.6f}、2026Q3现金{q["2026Q3"].cash_opportunity_log:+.6f}，不能由这两个已知季度设交易阈值。全部4年、16季及819连续126日窗口保存在Recorder。\n\n'
    f'组成候选的独立路径财富比{separate["full_relative_wealth"]:.6f}，其已封存分量另见原卡；两条账户的持仓、风险状态、费用路径不同，不将分量相减当组成映射的因果收益。决定`{summary["decision"]}`；原Stage1与组成候选均未通过全部SOXX硬门，后续仍须新决策时信息及完整0bp/2bp/5bp和独立前向。Recorder `{recorder_id}`，NAS SHA `{remote}`。\n'
)
print(report['full_relative_wealth'],report['full_log_components'],worst['first_trade'],worst['last_exit'],recorder_id)


0.6860064369263443 {'cash_opportunity_log': -0.6003226009036167, 'SOXL_selection_log': 0.22593145822639604, 'Stage1_direct_fees_log': -0.010264807879909826, 'timing_and_benchmark_residual_log': 0.007787682531410271} 2023-08-01 2024-01-31 bf012e010e3f4a5b9fd030b939ab989f


## 中文结论

原Stage1完整0bp账户相对SOXX财富比0.686006；全期对数超额−0.376868，现金机会−0.600323、SOXL相对选择+0.225931、直接费用−0.010265、显式时点残差+0.007788。最差126日现金−0.127238且SOXL选择−0.121381；2026Q3 SOXL选择−0.143901是主要损失。分量逐日闭合，但不同账户之间不能直接相减为因果收益。原生Qlib和NAS 11工件限定恢复通过；0拟合、0新账户，SOXX稳定超额仍未达成。